# רשתות עצביות חוזרות

במודול הקודם, כיסינו ייצוגים סמנטיים עשירים של טקסט. הארכיטקטורה שבה השתמשנו תופסת את המשמעות המצטברת של מילים במשפט, אך היא אינה מתחשבת ב**סדר** המילים, כיוון שהפעולה של האגרגציה שמתרחשת אחרי ההטמעות מסירה מידע זה מהטקסט המקורי. מכיוון שמודלים אלו אינם מסוגלים לייצג את סדר המילים, הם אינם יכולים לפתור משימות מורכבות או דו-משמעיות כמו יצירת טקסט או מענה על שאלות.

כדי לתפוס את משמעות רצף הטקסט, נשתמש בארכיטקטורה של רשת עצבית הנקראת **רשת עצבית חוזרת**, או RNN. כאשר משתמשים ב-RNN, אנחנו מעבירים את המשפט שלנו דרך הרשת טוקן אחד בכל פעם, והרשת מייצרת מצב **state** מסוים, שאותו אנו מעבירים שוב לרשת יחד עם הטוקן הבא.

![תמונה המראה דוגמה ליצירת רשת עצבית חוזרת.](../../../../../translated_images/he/rnn.27f5c29c53d727b5.webp)

בהינתן רצף הקלט של טוקנים $X_0,\dots,X_n$, ה-RNN יוצר רצף של בלוקי רשת עצבית, ומאתחל את הרצף הזה מקצה לקצה באמצעות למידה חוזרת לאחור (backpropagation). כל בלוק רשת מקבל זוג $(X_i,S_i)$ כקלט, ומייצר $S_{i+1}$ כתוצאה. המצב הסופי $S_n$ או הפלט $Y_n$ נשלחים למסווג ליניארי להפקת התוצאה. כל בלוקי הרשת משתפים את המשקולות זהות, ומאומנים מקצה לקצה באמצעות מעבר למידה חוזרת אחד.

> האיור למעלה מציג רשת עצבית חוזרת בצורה פרושה (משמאל), ובייצוג חוזר יותר מקוצר (מימין). חשוב להבין שכל תאי ה-RNN חולקים את אותם **משקולות ניתנות לשיתוף**.

מכיוון שמועברים וקטורי מצב $S_0,\dots,S_n$ דרך הרשת, ה-RNN יכול ללמוד תלות סדרתית בין מילים. לדוגמה, כאשר מופיעה המילה *not* איפשהו ברצף, הרשת יכולה ללמוד להכחיד אלמנטים מסוימים בתוך וקטור המצב.

מבפנים, כל תא RNN מכיל שתי מטריצות משקל: $W_H$ ו-$W_I$, ושיפוע $b$. בכל צעד RNN, בהינתן קלט $X_i$ ומצב קלט $S_i$, מצב הפלט מחושב כ-$S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$, כאשר $f$ היא פונקציית הפעלה (לעיתים קרובות $\tanh$).

> לבעיות כמו יצירת טקסט (שכיסוי יינתן ביחידה הבאה) או תרגום מכונה, אנו רוצים גם לקבל ערך פלט בכל צעד RNN. במקרה זה, ישנה גם מטריצת משקל נוספת $W_O$, והפלט מחושב כ-$Y_i=f(W_O\times S_i+b_O)$.

בואו נראה כיצד רשתות עצביות חוזרות יכולות לעזור לנו לסווג את מאגר החדשות שלנו.

> עבור סביבת הניסוי, עלינו להריץ את התא הבא כדי לוודא שהספרייה הדרושה מותקנת, והנתונים נקלטו מראש. אם אתם מריצים מקומית, תוכלו לדלג על התא הבא.


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

בעת אימון מודלים גדולים, הקצאת זיכרון GPU עלולה להפוך לבעיה. ייתכן שנרצה גם להתנסות בגדלים שונים של מיני-אצוות, כך שהנתונים יתאימו לזיכרון ה-GPU שלנו, ועדיין האימון יהיה מהיר דיו. אם אתה מריץ את הקוד הזה על מכונת GPU פרטית שלך, תוכל להתנסות בהתאמת גודל המיני-אצווה כדי להאיץ את האימון.

> **הערה**: ידוע כי גרסאות מסוימות של דרייברים של NVidia לא משחררות את הזיכרון לאחר אימון המודל. אנחנו מריצים כמה דוגמאות במחברת זו, וזה עשוי לגרום לזיכרון להיגמר בהתקנים מסוימים, במיוחד אם אתה מבצע ניסויים משלך כחלק מאותה מחברת. אם תיתקל בשגיאות מוזרות בעת התחלת אימון המודל, ייתכן שתרצה לאתחל את הקרנל של המחברת.


In [3]:
batch_size = 16
embed_size = 64

## מסווג RNN פשוט

במקרה של RNN פשוט, כל יחידת רשת חוזרת היא רשת ליניארית פשוטה, שלוקחת וקטור קלט ווקטור מצב, ומפיקה וקטור מצב חדש. ב-Keras, ניתן לייצג זאת בעזרת השכבה `SimpleRNN`.

אמנם ניתן להעביר אסימוני קידוד one-hot ישירות לשכבת ה-RNN, אך זו לא רעיון טוב בגלל הממד הגבוה שלהם. לכן, נשתמש בשכבת embedding להורדת הממד של וקטורי המילים, אחר כך בשכבת RNN, ולבסוף במסווג `Dense`.

> **הערה**: במקרים בהם הממד לא גבוה כל כך, למשל כשמשתמשים בטוקניזציה ברמת תווים, ייתכן שיהיה הגיוני להעביר אסימוני קידוד one-hot ישירות לתא ה-RNN.


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **הערה:** אנו משתמשים בשכבת הצמדה לא מאומנת כאן לפשטות, אך לתוצאות טובות יותר ניתן להשתמש בשכבת הצמדה מאומנת מראש באמצעות Word2Vec, כפי שמתואר ביחידה הקודמת. זו תהיה התנסות טובה עבורך להתאים את הקוד הזה לעבודה עם הצמדות מאומנות מראש.

עכשיו נאמן את ה-RNN שלנו. RNNים בכלל קשים למדי לאימון, כי ברגע שתאי ה-RNN מתפרסים לאורך האורך של הרצף, מספר השכבות המעורב ב-backpropagation הוא די גדול. לכן עלינו לבחור בקצב למידה קטן יותר, וללמד את הרשת על מערך נתונים גדול יותר כדי להפיק תוצאות טובות. זה יכול לקחת זמן רב, ולכן עדיף להשתמש ב-GPU.

כדי להאיץ את הדברים, נאמן את מודל ה-RNN רק על כותרות חדשות, בלי את התיאור. אתה יכול לנסות לאמן עם התיאור ולבדוק אם תצליח לאמן את המודל.


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> **שים לב** שהדיוק כנראה יהיה נמוך יותר כאן, מכיוון שאנו מתאמנים רק על כותרות חדשות.


## Revisiting variable sequences 

Remember that the `TextVectorization` layer will automatically pad sequences of variable length in a minibatch with pad tokens. It turns out that those tokens also take part in training, and they can complicate convergence of the model.

There are several approaches we can take to minimize the amount of padding. One of them is to reorder the dataset by sequence length and group all sequences by size. This can be done using the `tf.data.experimental.bucket_by_sequence_length` function (see [documentation](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

Another approach is to use **masking**. In Keras, some layers support additional input that shows which tokens should be taken into account when training. To incorporate masking into our model, we can either include a separate `Masking` layer ([docs](https://keras.io/api/layers/core_layers/masking/)), or we can specify the `mask_zero=True` parameter of our `Embedding` layer.

> **Note**: This training will take around 5 minutes to complete one epoch on the whole dataset. Feel free to interrupt training at any time if you run out of patience. What you can also do is limit the amount of data used for training, by adding `.take(...)` clause after `ds_train` and `ds_test` datasets.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


עכשיו כשאנחנו משתמשים בהסוואה, אנחנו יכולים לאמן את המודל על כל מערך הנתונים של הכותרים והתיאורים.

> **הערה**: האם שמתם לב שהשתמשנו בווקטורייזר שאומן על כותרות חדשות, ולא על כל גוף המאמר? ייתכן שזה יגרום לכך שחלק מהטוקנים יתעלמו, לכן עדיף לאמן מחדש את הווקטורייזר. עם זאת, זה עשוי להשפיע רק במעט, ולכן נמשיך להשתמש בווקטורייזר שאומן מראש מטעמי פשטות.


## LSTM: זיכרון לטווח קצר-ארוך

אחת הבעיות העיקריות של RNN היא **עלייה אפסית של הגרדיאנטים**. RNN יכולים להיות די ארוכים, וייתכן שיהיה להם קשה להפיץ את הגרדיאנטים כל הדרך חזרה לשכבה הראשונה של הרשת במהלך ההפצה לאחור. כאשר זה קורה, הרשת אינה יכולה ללמוד קשרים בין סמנים רחוקים. דרך אחת למנוע את הבעיה הזו היא להוסיף **ניהול מדויק של המצב** על ידי שימוש ב**שערים**. שתי הארכיטקטורות הנפוצות ביותר שמוסיפות שערים הן **זיכרון לטווח קצר-ארוך** (LSTM) ו**יחידת שידור ממוסגרת** (GRU). נעסוק כאן ב-LSTM.

![תמונה המציגה דוגמה לתא זיכרון לטווח קצר-ארוך](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

רשת LSTM מאורגנת בדומה ל-RNN, אך קיימים שני מצבים שמועברים משכבה לשכבה: המצב בפועל $c$, והווקטור הנסתר $h$. בכל יחידה, הווקטור הנסתר $h_{t-1}$ משולב עם הקלט $x_t$, וביחד הם שולטי מה קורה למצב $c_t$ ולפלט $h_{t}$ דרך **שערים**. לכל שער יש פונקציית הפעלה סיגמואידית (פלט בטווח $[0,1]$), שאפשר לחשוב עליה כמסכה ביטית כאשר היא מוכפלת בווקטור המצב. ל-LSTM יש את השערים הבאים (מימין לשמאל בתמונה למעלה):
* **שער השכחה** שמגדיר אילו רכיבים של הווקטור $c_{t-1}$ יש לשכוח ואילו להעביר הלאה.
* **שער הקלט** שמגדיר כמה מידע מהווקטור הקלט והווקטור הנסתר הקודם יש לשלב בווקטור המצב.
* **שער הפלט** שלוקח את ווקטור המצב החדש ומחליט אילו מהרכיבים שלו ישמשו ליצירת הווקטור הנסתר החדש $h_t$.

את רכיבי המצב $c$ אפשר לחשוב כעל דגלים שניתן להדליק ולכבות. לדוגמה, כשאנחנו נתקלים בשם *Alice* ברצף, אנו מנחשים שזה מתייחס לאישה, ומדליקים את הדגל במצב שאומר שיש שם עצם נקבי במשפט. כשנפגוש גם את המילים *and Tom*, נדליק את הדגל שאומר שיש שם עצם ברבים. כך על ידי ניהול המצב נוכל לעקוב אחרי התכונות הדקדוקיות של המשפט.

> **הערה**: הנה משאב מצוין להבנת הפנימיות של LSTM: [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) מאת כריסטופר אולה.

למרות שהמבנה הפנימי של תא LSTM עשוי להיראות מורכב, Keras מסתיר את המימוש הזה בתוך שכבת ה-`LSTM`, ולכן הדבר היחיד שעלינו לעשות בדוגמה שלמעלה הוא להחליף את השכבה החוזרת:


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> **שים לב** שאימון LSTMs גם הוא איטי למדי, וייתכן שלא תראה עלייה משמעותית בדיוק בתחילת האימון. ייתכן שתצטרך להמשיך לאמן זמן מה כדי להגיע לדיוק טוב.


## רשתות חוזרות דו-כיווניות ורב-שכבתיות (RNNs)

בדוגמאות שלנו עד כה, הרשתות החוזרות פועלות מהתחלת הרצף ועד סופו. זה מרגיש לנו טבעי כי זה עוקב אחרי הכיוון שבו אנחנו קוראים או מקשיבים לדיבור. עם זאת, בתרחישים הדורשים גישה אקראית לרצף הקלט, הגיוני יותר להריץ את החישוב החוזר בשני הכיוונים. רשתות חוזרות שמאפשרות חישובים בשני כיוונים נקראות **דו-כיווניות**, וניתן ליצור אותן על ידי עטיפת שכבת החזרה עם שכבת `Bidirectonal` מיוחדת.

> **הערה**: שכבת `Bidirectional` יוצרת שתי עותקים של השכבה בתוכה, ומגדירה את המאפיין `go_backwards` של אחד מהעותקים ל-`True`, מה שגורם לה לרוץ בכיוון ההפוך לאורך הרצף.

רשתות חוזרות, חד-כיווניות או דו-כיווניות, לוכדות תבניות בתוך רצף, ושומרות אותן בווקטורי מצב או מחזירות אותן כ-output. כמו עם רשתות קונבולוציה, ניתן לבנות שכבת חזרה נוספת לאחר הראשונה כדי ללכוד תבניות ברמה גבוהה יותר, שנבנו מתבניות ברמה נמוכה שזוהו על ידי השכבה הראשונה. זה מוביל למושג של **רשת חוזרת רב-שכבתית**, המורכבת משתי רשתות חוזרות או יותר, כאשר הפלט של השכבה הקודמת מועבר לשכבה הבאה כקלט.

![תמונה המציגה רשת LSTM רב-שכבתית לזיכרון לטווח ארוך](../../../../../translated_images/he/multi-layer-lstm.dd975e29bb2a59fe.webp)

*תמונה מתוך [מאמר נהדר זה](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3) מאת פרננדו לופז.*

Keras עושה את בניית הרשתות האלה למשימה קלה, כי כל מה שצריך זה להוסיף עוד שכבות חוזרות למודל. לכל השכבות חוץ מהאחרונה, יש צורך לציין את הפרמטר `return_sequences=True`, כי אנחנו רוצים שהשכבה תחזיר את כל מצבי הביניים, ולא רק את המצב הסופי של החישוב החוזר.

בואו בנבנה רשת LSTM דו-כיוונית בעלת שתי שכבות עבור בעיית הסיווג שלנו.

> **הערה**: קוד זה שוב לוקח זמן רב להשלמה, אבל הוא מעניק לנו את הדיוק הגבוה ביותר שראינו עד כה. אז אולי כדאי להמתין ולראות את התוצאה.


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## רשתות RNN למשימות נוספות

עד כה, התמקדנו בשימוש ברשתות RNN כדי לסווג רצפי טקסט. אך הן יכולות להתמודד עם משימות רבות נוספות, כמו יצירת טקסט ותרגום מכונה &mdash; נבחן את המשימות הללו ביחידה הבאה.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
